In [1]:
import pandas as pd

df = pd.read_csv("quality_data.csv")

# Create input text
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

df["input_text"] = df["input_text"].str.lower()

# Create label
df["label"] = df["category"] + "_" + df["subcategory"]

df = df[["input_text", "label"]]
df.head()

,input_text,label
0,narration: interest from fixed deposit | mode:...,investment_fd_interest
1,narration: fd interest payout | mode: neft | t...,investment_fd_interest
2,narration: fixed deposit interest | mode: neft...,investment_fd_interest
3,narration: bank interest received | mode: neft...,investment_fd_interest
4,narration: fd maturity interest | mode: neft |...,investment_fd_interest


In [2]:
from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = train_test_split(
#     df["input_text"],
#     df["label"],
#     test_size=0.2,
#     random_state=42,
#     stratify=df["label"]  # important for balance
# )
X_train = df["input_text"]
y_train = df["label"]

X_test = df["input_text"]
y_test = df["label"]

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

tfidf_model.fit(X_train, y_train)

y_pred_tfidf = tfidf_model.predict(X_test)

In [4]:
print(y_test.value_counts())

label
investment_fd_interest    10
expense_transport         10
investment_fd_booking      8
investment_stocks          8
income_salary              8
expense_shopping           8
expense_food               8
expense_bills              8
expense_insurance          8
expense_loan               8
expense_health             7
Name: count, dtype: int64


In [5]:
import numpy as np

probs = tfidf_model.predict_proba(X_test)
confidence_scores = np.max(probs, axis=1)

In [6]:
from gliner import GLiNER

gliner_model = GLiNER.from_pretrained("urchade/gliner_base")

labels = list(set(y_train))

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
def gliner_predict(text):
    entities = gliner_model.predict_entities(text, labels)

    if entities:
        return entities[0]["label"], entities[0]["score"]
    return "unknown", 0.0


y_pred_gliner = []
gliner_conf = []

for text in X_test:
    label, score = gliner_predict(text)
    y_pred_gliner.append(label)
    gliner_conf.append(score)

In [8]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_test, y_pred_tfidf, output_dict=True)

report_df = pd.DataFrame(report).transpose()

# Add F2 score manually
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)

report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)

report_df = report_df.round(3)

print(report_df)

                        precision  recall  f1-score  support  f2_score
expense_bills               0.875   0.875     0.875    8.000     0.875
expense_food                1.000   0.875     0.933    8.000     0.897
expense_health              1.000   1.000     1.000    7.000     1.000
expense_insurance           1.000   1.000     1.000    8.000     1.000
expense_loan                1.000   1.000     1.000    8.000     1.000
expense_shopping            1.000   1.000     1.000    8.000     1.000
expense_transport           0.909   1.000     0.952   10.000     0.980
income_salary               1.000   1.000     1.000    8.000     1.000
investment_fd_booking       1.000   1.000     1.000    8.000     1.000
investment_fd_interest      1.000   1.000     1.000   10.000     1.000
investment_stocks           1.000   1.000     1.000    8.000     1.000
accuracy                    0.978   0.978     0.978    0.978     0.978
macro avg                   0.980   0.977     0.978   91.000     0.978
weight

In [9]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_tfidf)
print("\nAccuracy:", round(accuracy, 3))


Accuracy: 0.978


In [10]:
print("\nBest Performing Category:")
print(report_df["f1-score"].idxmax(), report_df["f1-score"].max())

print("\nWorst Performing Category:")
print(report_df["f1-score"].idxmin(), report_df["f1-score"].min())


Best Performing Category:
expense_health 1.0

Worst Performing Category:
expense_bills 0.875


In [11]:
result_df = pd.DataFrame({
    "input_text": X_test,
    "actual_label": y_test,
    "predicted_label": y_pred_tfidf,
    "confidence": confidence_scores
}).reset_index(drop=True)

In [12]:
# Actual split
result_df[["category", "subcategory"]] = result_df["actual_label"].str.split("_", n=1, expand=True)

# Predicted split
result_df[["predicted_category", "predicted_subcategory"]] = result_df["predicted_label"].str.split("_", n=1, expand=True)

In [13]:
result_df["confidence_percent"] = (result_df["confidence"] * 100).round(2).astype(str) + "%"

def confidence_level(c):
    if c >= 0.7:
        return "High"
    elif c >= 0.4:
        return "Medium"
    else:
        return "Low"

result_df["confidence_level"] = result_df["confidence"].apply(confidence_level)

In [14]:
result_df["correct"] = result_df["actual_label"] == result_df["predicted_label"]

In [15]:
def extract_fields(text):
    parts = text.split("|")
    narration = parts[0].replace("narration:", "").strip()
    mode = parts[1].replace("mode:", "").strip()
    txn_type = parts[2].replace("type:", "").strip()
    return pd.Series([narration, mode, txn_type])

result_df[["narration", "mode", "type"]] = result_df["input_text"].apply(extract_fields)

In [16]:
final_df = result_df[
    [
        "narration",
        "mode",
        "type",
        "category",
        "subcategory",
        "predicted_category",
        "predicted_subcategory",
        "confidence",
        "confidence_percent",
        "confidence_level",
        "correct"
    ]
]

final_df.head(20)

,narration,mode,type,category,subcategory,predicted_category,predicted_subcategory,confidence,confidence_percent,confidence_level,correct
0,interest from fixed deposit,neft,credit,investment,fd_interest,investment,fd_interest,0.441928,44.19%,Medium,True
1,fd interest payout,neft,credit,investment,fd_interest,investment,fd_interest,0.464111,46.41%,Medium,True
2,fixed deposit interest,neft,credit,investment,fd_interest,investment,fd_interest,0.499856,49.99%,Medium,True
3,bank interest received,neft,credit,investment,fd_interest,investment,fd_interest,0.457348,45.73%,Medium,True
4,fd maturity interest,neft,credit,investment,fd_interest,investment,fd_interest,0.471337,47.13%,Medium,True
5,bank fd interest credit,neft,credit,investment,fd_interest,investment,fd_interest,0.516653,51.67%,Medium,True
6,fd interest received,neft,credit,investment,fd_interest,investment,fd_interest,0.518748,51.87%,Medium,True
7,roi amount credited,neft,credit,investment,fd_interest,investment,fd_interest,0.226884,22.69%,Low,True
8,term deposit interest,neft,credit,investment,fd_interest,investment,fd_interest,0.456984,45.7%,Medium,True
9,fd interest deposit,neft,credit,investment,fd_interest,investment,fd_interest,0.567799,56.78%,Medium,True


In [17]:
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred_tfidf, output_dict=True)
report_df = pd.DataFrame(report).transpose()

In [18]:
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)

report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)

report_df = report_df.round(3)

print("\nClassification Report:")
print(report_df)


Classification Report:
                        precision  recall  f1-score  support  f2_score
expense_bills               0.875   0.875     0.875    8.000     0.875
expense_food                1.000   0.875     0.933    8.000     0.897
expense_health              1.000   1.000     1.000    7.000     1.000
expense_insurance           1.000   1.000     1.000    8.000     1.000
expense_loan                1.000   1.000     1.000    8.000     1.000
expense_shopping            1.000   1.000     1.000    8.000     1.000
expense_transport           0.909   1.000     0.952   10.000     0.980
income_salary               1.000   1.000     1.000    8.000     1.000
investment_fd_booking       1.000   1.000     1.000    8.000     1.000
investment_fd_interest      1.000   1.000     1.000   10.000     1.000
investment_stocks           1.000   1.000     1.000    8.000     1.000
accuracy                    0.978   0.978     0.978    0.978     0.978
macro avg                   0.980   0.977     0.978  

In [19]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_tfidf)

print("\nOverall Metrics:")
print("Accuracy:", round(accuracy, 3))
print("Macro F1:", report_df.loc["macro avg", "f1-score"])
print("Weighted F1:", report_df.loc["weighted avg", "f1-score"])


Overall Metrics:
Accuracy: 0.978
Macro F1: 0.978
Weighted F1: 0.978


In [20]:
# Remove summary rows
filtered_df = report_df.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")

best = filtered_df["f1-score"].idxmax()
worst = filtered_df["f1-score"].idxmin()

print("\nBest Performing Category:")
print(best, filtered_df.loc[best, "f1-score"])

print("\nWorst Performing Category:")
print(worst, filtered_df.loc[worst, "f1-score"])


Best Performing Category:
expense_health 1.0

Worst Performing Category:
expense_bills 0.875
